In [1]:
%load_ext autoreload
%autoreload 2

from config import SimConfig, HybridSimConfig
from src.plants.fc_only_plant import FuelCellOnlyPlant
from src.plants.hybrid_plant import FuelCellBatteryPlant
from src.data_processing import load_and_cache_entire_fleet
from src.utils.evaluation import VoyageBenchmarker
from src.plotting import plot_dashboard

# Initialize Configs and Plants
cfg_base = SimConfig()
plant_base = FuelCellOnlyPlant(cfg_base)

cfg_hybrid = HybridSimConfig(lambda_scale=60, dt=5.0)
plant_hybrid = FuelCellBatteryPlant(cfg_hybrid)

# Cache data to RAM once
fleet_cache = load_and_cache_entire_fleet(cfg_base)

# Initialize our new Orchestrator (Excluding bad sensor days 1, 2, and 3)
benchmarker = VoyageBenchmarker(fleet_cache, exclude_days=[1, 2, 3])

Beginning memory staging of all 14 fleet files into RAM...
 -> Day 01 successfully cached in RAM.
 -> Day 02 successfully cached in RAM.
 -> Day 03 successfully cached in RAM.
 -> Day 04 successfully cached in RAM.
 -> Day 05 successfully cached in RAM.
 -> Day 06 successfully cached in RAM.
 -> Day 07 successfully cached in RAM.
 -> Day 08 successfully cached in RAM.
 -> Day 09 successfully cached in RAM.
 -> Day 10 successfully cached in RAM.
 -> Day 11 successfully cached in RAM.
 -> Day 12 successfully cached in RAM.
 -> Day 13 successfully cached in RAM.
 -> Day 14 successfully cached in RAM.

All 14 operational days securely held in RAM. Disk I/O locked.


In [2]:
# Create declarative definitions for whatever you want to test
baseline_heuristic = {
    "name": "Baseline Heuristic", "is_hybrid": False, "strategy": "HEURISTIC",
    "config": cfg_base, "plant": plant_base
}

baseline_sdp = {
    "name": "Baseline SDP", "is_hybrid": False, "strategy": "SDP",
    "config": cfg_base, "plant": plant_base
}

hybrid_heuristic = {
    "name": "Hybrid Heuristic", "is_hybrid": True, "strategy": "HEURISTIC",
    "config": cfg_hybrid, "plant": plant_hybrid
}

hybrid_tensor_sdp = {
    "name": "Hybrid Tensor SDP", "is_hybrid": True, "strategy": "SDP",
    "sdp_variant": "TENSOR_SWEEP", "config": cfg_hybrid, "plant": plant_hybrid
}

hybrid_mean_sdp = {
    "name": "Hybrid Mean SDP", "is_hybrid": True, "strategy": "SDP",
    "sdp_variant": "MEAN_PROXY", "config": cfg_hybrid, "plant": plant_hybrid
}

In [3]:
# Compare multiple strategies head-to-head
approaches_to_compare = {
    "Baseline Heuristic": baseline_heuristic,
    "Baseline Optimized": baseline_sdp,
    "Hybrid Heuristic": hybrid_heuristic,
    "Hybrid Optimized (Tensor)": hybrid_tensor_sdp,
    "Hybrid Optimized (Mean)": hybrid_mean_sdp
}

# Train on Days 4 through 10, Test on Day 12
df_comparison, sims = benchmarker.compare_approaches(
    approaches_to_compare, 
    train_days=[4, 5, 6, 7, 8, 9, 10, 11, 12, 13], 
    test_day=14
)

display(df_comparison)


Comparing 5 approaches | Train: [4, 5, 6, 7, 8, 9, 10, 11, 12, 13] | Test: Day 14
 -> Running: Baseline Heuristic
 -> Running: Baseline Optimized
 -> Running: Hybrid Heuristic
 -> Running: Hybrid Optimized (Tensor)
 -> Launching TENSOR_SWEEP Solver...
 -> Triggering Offline MC Tensor Pre-Computation (25 paths)...
 -> Tensors Cached. Initiating O(M^2) Online Sweep...
 -> Running: Hybrid Optimized (Mean)
 -> Launching MEAN_PROXY Solver...


,Total Cost ($),H2 Cost ($),FC Switch Cost ($),Bat. Degrade ($),Final SoC (%),Compute Time (s)
Baseline Heuristic,104.656666,34.656666,70.0,0.000000,NaN,2.220105
Baseline Optimized,174.536865,65.536865,109.0,0.000000,NaN,7.986281
Hybrid Heuristic,412.864901,246.722552,73.0,93.142349,49.558615,0.683021
Hybrid Optimized (Tensor),284.852416,69.106005,97.0,118.746410,56.192979,4.952698
Hybrid Optimized (Mean),292.124576,70.105093,124.0,98.019484,56.599865,6.442280


In [7]:
# Plot the Baseline SDP
# plot_dashboard(sims['Baseline Heuristic'], 'Baseline Heuristic', test_day=14, layout='grid')
# plot_dashboard(sims['Baseline Optimized'], 'Baseline Optimized', test_day=14, layout='grid')

# Plot the Hybrid comparison dashboard for the best performing approach
# plot_dashboard(sims["Hybrid Heuristic"], "Hybrid Heuristic", test_day=14, layout='grid')
# plot_dashboard(sims["Hybrid Optimized (Tensor)"], "Hybrid Optimized (Tensor)", test_day=14, layout='grid')
plot_dashboard(sims["Hybrid Optimized (Mean)"], "Hybrid Optimized (Mean)", test_day=14, layout='grid')

In [ ]:
# Run Chronological Forward Chaining for the Hybrid Mean SDP
print("--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---")
df_forward = benchmarker.run_forward_chaining(hybrid_mean_sdp, min_train_days=1)
display(df_forward)

# Run Leave-One-Out for the Baseline SDP
print("\n--- APPROACH A: LEAVE ONE OUT ---")
df_loo = benchmarker.run_leave_one_out(hybrid_mean_sdp)
display(df_loo)

# You can now plot df_forward['Total Cost ($)'] just like you did in your original notebook!

--- APPROACH B: CHRONOLOGICAL FORWARD CHAINING ---

Starting Forward Chaining CV for: Hybrid Mean SDP
 -> Chaining Step 1: Training on [4] | Testing on 5
 -> Launching MEAN_PROXY Solver...


SystemError: CPUDispatcher(<function _solve_mean_proxy_bellman at 0x0000022AF23B85E0>) returned a result with an exception set

In [ ]:
# Compare multiple strategies head-to-head
approaches_to_compare = {
    "Baseline Rule-Based": baseline_heuristic,
    "Baseline Optimized": baseline_sdp,
    "Hybrid Optimized (Tensor)": hybrid_tensor_sdp,
    "Hybrid Optimized (Mean)": hybrid_mean_sdp
}

# Train on Days 4 through 10, Test on Day 12
df_comparison = benchmarker.compare_approaches(
    approaches_to_compare, 
    train_days=[4, 5, 6, 7, 8, 9, 10, 11, 12, 13], 
    test_day=14
)

display(df_comparison)